# Raw Subthreshold Voltage Save Patch

This notebook contains drop-in replacement cells for the main simulation notebook so that each recording saves both visualization voltage traces and raw sampled membrane voltage.

Use it as a patch notebook:
1. Copy Cell 2 into Part 4 of the main simulation notebook to replace `simulate_network()`.
2. Copy Cell 3 into Part 6 to replace `save_recording_data()`.

The new raw trace key is `voltage_traces_raw`, which stores the sampled membrane variable without the artificial +20 mV spike peaks used for visualization.

In [ ]:
import numpy as np

def simulate_network(neurons, synapses, stimulation_events, dt=0.1, duration=1000,
                     record_voltage=True, voltage_sample_rate=1.0):
    """
    Simulate network activity over time.

    Returns:
        spike_data: Dictionary mapping neuron_id to list of spike times
        voltage_data: Dictionary with visualization and raw voltage traces
    """
    print(f"\nSimulating {duration} ms of network activity...")
    print("  Mode: Stimulus-driven bursting")

    num_steps = int(duration / dt)
    t = 0

    synapse_lookup = {}
    for syn in synapses:
        synapse_lookup.setdefault(syn.pre_neuron_id, []).append(syn)
    print(f"  Built synapse lookup: {len(synapse_lookup)} presynaptic neurons")

    spike_data = {neuron.neuron_id: [] for neuron in neurons}

    voltage_data = None
    if record_voltage:
        voltage_sample_steps = int(voltage_sample_rate / dt)
        n_voltage_samples = int(duration / voltage_sample_rate)
        voltage_traces = np.zeros((len(neurons), n_voltage_samples))
        voltage_traces_raw = np.zeros((len(neurons), n_voltage_samples))
        voltage_times = np.arange(n_voltage_samples) * voltage_sample_rate
        voltage_sample_idx = 0

    active_stims = {}
    stim_idx = 0

    spiked_since_last_sample = set()
    SPIKE_PEAK_MV = 20.0

    for step in range(num_steps):
        t = step * dt

        if step % (num_steps // 10) == 0:
            print(f"  Progress: {100 * step / num_steps:.0f}%")

        while stim_idx < len(stimulation_events) and stimulation_events[stim_idx][0] <= t:
            stim_t, neuron_id, amplitude, duration_stim = stimulation_events[stim_idx]
            active_stims[neuron_id] = (amplitude, stim_t + duration_stim)
            stim_idx += 1

        expired = [nid for nid, (amp, end_t) in active_stims.items() if t > end_t]
        for nid in expired:
            del active_stims[nid]

        for neuron in neurons:
            if neuron.neuron_id in active_stims:
                neuron.i_ext = active_stims[neuron.neuron_id][0]
            else:
                neuron.i_ext = 0.0

        for neuron in neurons:
            neuron.i_syn = 0.0

        for syn in synapses:
            i_syn = syn.update(t, dt)
            syn.post_neuron.i_syn += i_syn

        for neuron in neurons:
            spiked = neuron.update(t, dt)

            if spiked:
                spike_data[neuron.neuron_id].append(t)
                spiked_since_last_sample.add(neuron.neuron_id)

                if neuron.neuron_id in synapse_lookup:
                    for syn in synapse_lookup[neuron.neuron_id]:
                        syn.receive_spike(t)

        if record_voltage and step % voltage_sample_steps == 0:
            if voltage_sample_idx < n_voltage_samples:
                for neuron in neurons:
                    voltage_traces_raw[neuron.neuron_id, voltage_sample_idx] = neuron.v
                    if neuron.neuron_id in spiked_since_last_sample:
                        voltage_traces[neuron.neuron_id, voltage_sample_idx] = SPIKE_PEAK_MV
                    else:
                        voltage_traces[neuron.neuron_id, voltage_sample_idx] = neuron.v
                voltage_sample_idx += 1
                spiked_since_last_sample.clear()

    print("  Progress: 100%")
    print("Simulation complete!\n")

    if record_voltage:
        voltage_data = {
            'traces': voltage_traces,
            'traces_raw': voltage_traces_raw,
            'times': voltage_times,
            'sample_rate': voltage_sample_rate,
        }

    return spike_data, voltage_data

In [ ]:
import os
import numpy as np

def save_recording_data(spike_data, voltage_data, cluster_info, recording_idx,
                       timestamp, save_dir, target_freq=10, duration=60000,
                       burst_onset_times=None):
    """
    Save recording data to file (spikes, visualization voltage, raw voltage, and resampled data).
    """
    session_dir = os.path.join(save_dir, timestamp)
    os.makedirs(session_dir, exist_ok=True)

    filename = os.path.join(session_dir, f'recording{recording_idx:03d}.npz')

    spike_times_list = np.array([spike_data[i] for i in range(len(spike_data))], dtype=object)
    cluster_spike_data = organize_spike_data_by_cluster(spike_data, cluster_info)
    cluster_spike_data = np.array([np.array(cluster, dtype=object) for cluster in cluster_spike_data], dtype=object)

    resampled_spikes, resampled_time_points, resampled_spike_positions = resample_data(
        spike_data, cluster_info['cluster_assignments'], target_freq, duration
    )

    save_dict = {
        'spike_times': spike_times_list,
        'cluster_spike_data': cluster_spike_data,
        'resampled_spikes': resampled_spikes,
        'resampled_time_points': resampled_time_points,
        'resampled_cluster_assignments': cluster_info['cluster_assignments'],
        'resampling_frequency': target_freq,
        'resampling_interval_ms': 1000.0 / target_freq,
        'resampled_spike_positions': resampled_spike_positions,
        'recording_index': recording_idx,
        'timestamp': timestamp,
        'duration': duration,
    }

    if voltage_data is not None:
        save_dict['voltage_traces'] = voltage_data['traces']
        save_dict['voltage_traces_raw'] = voltage_data['traces_raw']
        save_dict['voltage_times'] = voltage_data['times']
        save_dict['voltage_sample_rate'] = voltage_data['sample_rate']

    if burst_onset_times is not None:
        save_dict['burst_onset_times'] = np.array(burst_onset_times)

    np.savez_compressed(filename, **save_dict)

    print(f"Recording {recording_idx} saved to: {filename}")
    print(f"  - Original spike times: {len(spike_times_list)} neurons")
    print(f"  - Resampled data: {resampled_spikes.shape[0]} neurons x {resampled_spikes.shape[1]} time points")
    if voltage_data is not None:
        print(f"  - Voltage traces: {voltage_data['traces'].shape[0]} neurons x {voltage_data['traces'].shape[1]} time points")
        print(f"  - Raw voltage traces: {voltage_data['traces_raw'].shape[0]} neurons x {voltage_data['traces_raw'].shape[1]} time points")
    if burst_onset_times is not None:
        print(f"  - Burst onset times: {len(burst_onset_times)} stimuli")
    return filename

## New saved keys

- `voltage_traces`: visualization voltage with artificial spike peaks
- `voltage_traces_raw`: raw sampled membrane voltage without artificial peaks
- `voltage_times`
- `voltage_sample_rate`